In [ ]:
%pip install anthropic python-dotenv

In [22]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
client = Anthropic()

model = "claude-sonnet-4-6"

In [ ]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

In [ ]:
message

In [ ]:
print(message.content[0].text)

In [23]:
# user message helper function
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

# assistant message helper function
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# chat helper function
def chat(messages):
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages
    )
    return response.content[0].text

In [ ]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

### **CHAT BOT**

In [ ]:
chat_messages = []

while True:
    user_input = input("")
    print(">", user_input)

    add_user_message(chat_messages, user_input)

    response = chat(chat_messages)

    add_assistant_message(chat_messages, response)
    print(response)
    print("--------")

### **SYSTEM PROMPTS**

In [24]:
def chat_with_sys_prompt(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    
    if system:
        params["system"] = system
    
    response = client.messages.create(**params)

    return response.content[0].text

In [20]:
messages = []

prompt = "Solve 2x + 10 = 5 for x?"

add_user_message(messages, prompt)

sys_prompt = """
        You are a patient math tutor.
        Do not directly answer a student's questions.
        Guide them to a solution step by step.
    """

answer = chat_with_sys_prompt(messages)

print(answer)

## Solving for x

**Given equation:** 2x + 10 = 5

**Step 1: Subtract 10 from both sides**

2x + 10 - 10 = 5 - 10

2x = -5

**Step 2: Divide both sides by 2**

2x/2 = -5/2

**x = -2.5** (or -5/2)

### Verification
2(-2.5) + 10 = -5 + 10 = **5** ✓


In [25]:
messages = []

prompt = "Write a python function that checks a string for duplicate characters"

add_user_message(messages, prompt)

sys_prompt = """
        You are a Python engineer who writes very concise and short code.
    """

answer = chat_with_sys_prompt(messages, system=sys_prompt)

print(answer)

```python
def has_duplicates(s):
    return len(s) != len(set(s))
```

**Example usage:**
```python
print(has_duplicates("hello"))   # True  ('l' appears twice)
print(has_duplicates("world"))   # False
```

If you also want to **find which characters are duplicated**:

```python
def find_duplicates(s):
    return {c for c in s if s.count(c) > 1}

print(find_duplicates("hello"))  # {'l'}
```


### **TEMPERATURE**

In [33]:
def chat_with_temp(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    
    response = client.messages.create(**params)

    return response.content[0].text

In [39]:
messages = []

prompt = "Generate a one sentence movie idea for me"

add_user_message(messages, prompt)

response = chat_with_temp(messages)

print(response)

Here's a movie idea:

A seasoned lighthouse keeper discovers that the mysterious light appearing across the ocean each night is actually a signal from a parallel world on the brink of collapse, and she must decode its messages before both worlds are destroyed.


### **RESPONSE STREAMING**

In [40]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01W5PSG9Lmdz8vwVAmJrxohc', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='Here', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' is a 1 sentence description of a fake database:\n\n**NebulaDB** is a fictional cloud-based rel', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(

In [47]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")


final_message = stream.get_final_message()
print(final_message.content[0].text)

Here is a one sentence description of a fake database:

**"The Galactic Registry of Fictional Beings"** is a mock database containing over 10,000 fabricated records of imaginary creatures, including their made-up names, invented home planets, and nonsensical biological classifications.Here is a one sentence description of a fake database:

**"The Galactic Registry of Fictional Beings"** is a mock database containing over 10,000 fabricated records of imaginary creatures, including their made-up names, invented home planets, and nonsensical biological classifications.
